# Logistic Regression using Scikit-Learn
-  Train a logistic regression model using scikit-learn.

## What's in this notebook

- Fit `LogisticRegression`
- Predict
- Accuracy
- Reading the coefficients: log-odds and odds ratios
- More than two classes: softmax vs one-vs-rest

## Dataset 
Let's start with the same dataset as before.

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# Dataset
x_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y = np.array([0, 0, 0, 1, 1, 1])

# Fit Model
lr_model = LogisticRegression()
lr_model.fit(x_train, y)

# Make Prediction
y_predict = lr_model.predict(x_train)
print("Predictions on training set: ", y_predict)

# Calculate Accuracy
print("Accuracy on training set: ", lr_model.score(x_train,y))




Predictions on training set:  [0 0 0 1 1 1]
Accuracy on training set:  1.0


# Reading the Coefficients: Log-odds and Odds Ratios

Logistic regression is linear in the **log-odds**: `log(p / (1 − p)) = w·x + b`. So for feature j:
- `wⱼ` is the change in log-odds when the feature goes up by 1, with the other features fixed
- `e^wⱼ` is the **odds ratio**: the odds get multiplied by `e^wⱼ`
- with standardized features, "+1" means "+1 standard deviation", so coefficients can be compared with each other

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=42)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)

coef = pd.Series(clf[-1].coef_[0], index=data.feature_names)
top = coef[coef.abs().sort_values(ascending=False).index[:3]]
print("classes:", [str(c) for c in data.target_names], "-> the model predicts the odds of", data.target_names[1])
print(pd.DataFrame({"weight w": top.round(3), "odds ratio e^w": np.exp(top).round(3)}))


classes: ['malignant', 'benign'] -> the model predicts the odds of benign
                weight w  odds ratio e^w
worst texture     -1.250           0.286
radius error      -1.070           0.343
worst symmetry    -0.957           0.384


Class 1 is "benign", so the model gives the odds of benign. "worst texture" has the largest weight, −1.250: one standard deviation more texture
multiplies the odds of benign by e^−1.25 = 0.286, about 71% lower odds, with the other features fixed.

Read coefficients with care:
- correlated features share weight (radius, perimeter and area here), so one coefficient on its own can be misleading
- regularization (`C`) shrinks all of them
- a coefficient shows association in this data, not cause and effect

# More Than Two Classes: Softmax vs One-vs-Rest

- **One-vs-rest (OvR):** train K binary models, each "class k vs everything else", and predict the class with the highest score.
- **Softmax (multinomial):** one model with K weight vectors, trained together with cross-entropy: `P(class k) = e^zₖ / Σⱼ e^zⱼ`.
  The probabilities sum to 1 by design. sklearn's `LogisticRegression` uses softmax for multi-class with its default solver.

In [2]:

from sklearn.multiclass import OneVsRestClassifier

X_iris, y_iris = load_iris(return_X_y=True)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(X_iris, y_iris, test_size=0.3, stratify=y_iris, random_state=0)

softmax = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xi_tr, yi_tr)
ovr = make_pipeline(StandardScaler(), OneVsRestClassifier(LogisticRegression(max_iter=1000))).fit(Xi_tr, yi_tr)
print(f"softmax:      test accuracy {softmax.score(Xi_te, yi_te):.3f}, probabilities for one flower {softmax.predict_proba(Xi_te[:1]).round(3)}")
print(f"one-vs-rest:  test accuracy {ovr.score(Xi_te, yi_te):.3f}, probabilities for one flower {ovr.predict_proba(Xi_te[:1]).round(3)}, "
      f"from {len(ovr[-1].estimators_)} separate models")


softmax:      test accuracy 0.978, probabilities for one flower [[0.    0.036 0.964]]
one-vs-rest:  test accuracy 0.889, probabilities for one flower [[0.002 0.088 0.91 ]], from 3 separate models


Softmax scores 0.978 on the iris test set and one-vs-rest 0.889. The 3 one-vs-rest models never compete during training:
"versicolor vs rest" is a hard binary problem on its own, and its mistakes show up at prediction time.

| | One-vs-rest | Softmax (multinomial) |
|---|---|---|
| Models | K separate binary models | one model with K weight vectors, trained together |
| Probabilities | rescaled to sum to 1 afterwards; not trained jointly | sum to 1 by design; usually better calibrated |
| Gain | works with any binary classifier; the K models train in parallel | usually more accurate; one fit |
| Cost | the classes only compete at prediction time | adding a class means retraining everything |
| Used by | `LinearSVC`, the `liblinear` solver | `LogisticRegression` (default solver), neural network output layers |